In [14]:
#Imports necessários

import pandas as pd
import numpy as np

import mlflow
import mlflow.sklearn

from sklearn.model_selection import (
    train_test_split,
    cross_val_score
)

from sklearn.naive_bayes import GaussianNB

from sklearn.metrics import (
    accuracy_score
)

from sklearn.preprocessing import LabelEncoder

In [15]:
#Configuaração do Mlflow


mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("Naive_Bayes")

mlflow.sklearn.autolog()

In [16]:
#Carregar o dataset

df = pd.read_csv("../DATASET/dataset_expandido.csv")

In [17]:
#Preparação dos dados

df["YearsCodePro_Num"] = (df["YearsCodePro"])

df["YearsCodePro_Num"] = (df["YearsCodePro_Num"].replace({"Less than 1 year":0, "More than 50 years":51, "Sem dado":0}))

df["YearsCodePro_Num"] = pd.to_numeric(df["YearsCodePro_Num"])

features = ["YearsCodePro_Num", "WorkExp", "Age_Code"]

target = "JobSat_Class"

X = df[features]

y = df[target]

In [18]:
#Preparação da função do naive bayes

def run_nb_experiment(run_name, X, y, random_state=42):

    X_train, X_test, y_train, y_test = (train_test_split(X, y, test_size=0.2, random_state=random_state))

    with mlflow.start_run(run_name=run_name):

        model = GaussianNB()

        model.fit(X_train, y_train)

        y_pred = model.predict(X_test)

        accuracy = accuracy_score(y_test, y_pred)

        cv = cross_val_score(model, X, y, cv=5)

        mlflow.log_param("features", str(list(X.columns)))

        mlflow.log_param("random_state", random_state)

        mlflow.log_metric("accuracy", accuracy)


        mlflow.log_metric("cv_mean", cv.mean())

        mlflow.log_metric("cv_std", cv.std())

        print(run_name)

        print("Accuracy:", accuracy)

        print("CV Mean:", cv.mean())

        print("CV Std:", cv.std())

        print("-"*40)

In [19]:
#Experiencia 0 - Baseline

run_nb_experiment(

    run_name= "Experiment_0_Baseline",
    X=X,
    y=y
)

2026/05/23 17:19:29 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/23 17:19:29 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_0_Baseline
Accuracy: 0.7311109767891683
CV Mean: 0.7316017571253888
CV Std: 0.004746657563100255
----------------------------------------


In [20]:
#Experiencias 1- Adição de mais features


features_exp1 = [

    "YearsCodePro_Num",

    "WorkExp",

    "Age_Code",

    "JobSatPoints_1",

    "JobSatPoints_4",

    "JobSatPoints_5",

    "JobSatPoints_6",

    "JobSatPoints_7",

    "JobSatPoints_8",

    "JobSatPoints_9"

]

X_exp1 = df[features_exp1]

run_nb_experiment(

    run_name=
    "Experiment_1_Add_Features",
    X=X_exp1,
    y=y
)

2026/05/23 17:19:39 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/23 17:19:39 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_1_Add_Features
Accuracy: 0.7337705512572534
CV Mean: 0.7310879745979462
CV Std: 0.007174578694592219
----------------------------------------


In [21]:
#Experiencia 2A - Remoção de Valores em Falta

df_drop = df.dropna()

X_drop = df_drop[
    features
]

y_drop = df_drop[target]

run_nb_experiment(

    run_name=
    "Experiment_2_Drop_Missing",
    X=X_drop,
    y=y_drop
)

2026/05/23 17:19:47 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/23 17:19:48 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_2_Drop_Missing
Accuracy: 0.7311109767891683
CV Mean: 0.7316017571253888
CV Std: 0.004746657563100255
----------------------------------------


In [22]:
# Experiencia 2B - Substituição de Valores em Falta pela Média

df_imp = df.copy()

for col in features:

    df_imp[col] = (
        df_imp[col]
        .fillna(
            df_imp[col]
            .mean()
        )
    )

X_imp = df_imp[
    features
]

y_imp = df_imp[
    target
]

run_nb_experiment(

    run_name=
    "Experiment_3_Imputation",

    X=X_imp,

    y=y_imp
)

2026/05/23 17:19:58 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/23 17:19:59 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_3_Imputation
Accuracy: 0.7311109767891683
CV Mean: 0.7316017571253888
CV Std: 0.004746657563100255
----------------------------------------


In [ ]:
#Experiencia 3 - Adicionar Variáveis Categóricas


df_cat = df.copy()

encoder = LabelEncoder()

df_cat["MainBranch_Code"] = encoder.fit_transform(df_cat["MainBranch"])

df_cat["RemoteWork_Code"] = encoder.fit_transform(df_cat["RemoteWork"])

features_cat = [
    "YearsCodePro_Num",
    "WorkExp",
    "Age_Code",
    "MainBranch_Code",
    "RemoteWork_Code"
]

X_cat = df_cat[features_cat]

run_nb_experiment(
    run_name=
    "Experiment_4_Categorical",
    X=X_cat,
    y=y
)

2026/05/23 17:20:08 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/23 17:20:08 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_4_Categorical
Accuracy: 0.7270309477756286
CV Mean: 0.7276848563812103
CV Std: 0.006733168522699819
----------------------------------------


In [24]:
#Experiencia 4A - Alteração do Random State (7)

run_nb_experiment(

    run_name=
    "Experiment_5_Random_7",
    X=X,
    y=y,
    random_state=7
)

2026/05/23 17:20:17 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/23 17:20:17 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_5_Random_7
Accuracy: 0.7309296421663443
CV Mean: 0.7316017571253888
CV Std: 0.004746657563100255
----------------------------------------


In [25]:
#Experiencia 4A - Alteração do Random State (123)

run_nb_experiment(

    run_name=
    "Experiment_6_Random_123",
    X=X,
    y=y,
    random_state=123
)

2026/05/23 17:20:26 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/23 17:20:26 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_6_Random_123
Accuracy: 0.730446083172147
CV Mean: 0.7316017571253888
CV Std: 0.004746657563100255
----------------------------------------


In [26]:
#Experiencia 5 - Seleção de Features

features_small = [

    "YearsCodePro_Num",
    "WorkExp"

]

X_small = df[features_small]

run_nb_experiment(

    run_name=
    "Experiment_7_Feature_Selection",
    X=X_small,
    y=y
)

2026/05/23 17:20:33 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Local\Programs\Python\Python313\Lib\site-packages\mlflow\types\utils.py:452: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Values <https://www.mlflow.org/docs/latest/models.html#handling-integers-with-missing-values>`_ for more details."
2026/05/23 17:20:33 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "c:\Users\luisr\AppData\Lo

Experiment_7_Feature_Selection
Accuracy: 0.730536750483559
CV Mean: 0.7309126778859018
CV Std: 0.005015121784257344
----------------------------------------


### Introdução

algoritmo Gaussian Naive Bayes foi utilizado para a classificação da variàvel JobSat_Class. Foram utilizadas as mesmas variáveis (Years_CodePro_Num, WorkExp e Age_Code). Foram efetuadas várias experiências como:

    . Aumento do número de features;
    . Tratamento de valores em falta;
    . Inclusão de variáveis categóricas;
    . Alteração da Aleatoriedade;

De uma forma geral, dá para perceber que o modelo apresentou um desempenho estável, onde o valor da accuracy anda a rondar os 73%, o que indica que o modelo possui uma capacidade moderada para prever a satisfação profissional.

### ANÁLISE DE EXPERIÊNCIAS

# Experiência 0 - Naive Bayes Baseline

O modelo apresentou uma accuracy de aproximadamente 73%. Este resultado demonstra que mesmo utilizando poucas variáveis o Naive Bayes conseguiu capturar relações relevantes para a classificação.
Além disso o Desvio Padrão da Cross Validation obteve valores de cerca de 0.005, o que mostra estabilidade.

# Experiência 1 - Adição de Features

Nesta esperiência o objetivo era adicionbar novas variáveis, foram adicionadas variáveis relacionadas com o JobSatPoints. Aqui o objetivo era verificar se o aumento da dimensionalidade melhroava a classificação. 

Os resultados obtidos mostraram que existiu uma mnelhora do desempenho, ou seja, as novas variáveis introduziram nova informação que foi útil para o modelo. Apesar disso a melhoria foi relativamente pequena, ou seja, grande parte da informação acrescentada, já estava representada pelas variáveis iniciais.

# Experiência 2A - Remoção de Valores em Falta

Nesta experiencia, no codigo utilizamos o 
        .dropna(), para efetuar a remoção completa das linhas que possuiam valores ausentes. Verificando os resultados, percebemos que o impacto foi quase inexistente.

ISto acontece porque o tratamento do dataset já tinha ocorrido, inclusive na 1º parte do trabalho, onde foi efetuada essa verificação, para a remoção de valores em falta ou valores NaN. Assim, a remoção não alterou de forma relevante a distribuição dos dados.

# Experiência 2B - Substituição pela Média

Neste caso, foi praticamente igual à experiência anterior, com a exceção que não ouve remoção dos valores ausentes, mas sim, a sua substituição pela média, utilizando o seguinte código:

    .fillna(mean)

Tal como esperado, os resultados permaneceram praticamente inalterados. Ou seja, o tartamento do dataset que efetuamos anteriormente, já tinha resolvido estes problemas e portanto estas duas experiências tiveram um efeito reduzido.


# Experiência 3 - Inclusão de Variáveis Categóricas

Esta foi uma das experiência mais interessantes, porque foram adicionadas variàveis categóricas. Nesta experiência houve uma redução do desempenho, tendo uma accuracy de aproximadamente 72,8% e um aumento da variabilidade CV std = 0.0067.

O que retiramos como conclusão é que as variáveis categóricas adicionadas não introduziram informação suficientemente relevante e podem ter aumentado ruído no modelo.

# Experiência 4 - Alteração do Random State

Para esta experiência foram testados dois valors, um baixo (7) e outro mais alto (123). Os resultados foram praticamnete identicos, logo o modelo possui uma boa estabilidade e tambem que não depende muito da divisão aleatória dos dados. Isto é bom, porque significa que o modelo tem uma boa capacidade de generalização.

# Experiência 5 - Seleção de Features

Nesta experienci reduzimos o conjunto para apenas duas features, e mesmo reduzinfo a dimensionalidade, o desempenho manteve-se próximo das outras experiência com uma accuracy de 73% aproximadamente.

Ou seja, isto significa que a variável removida, no caso a Age_Code possui um impacto reduzido na classificação da satisfação profissional. E verificamos tambem que o modelo consegue funcionar bem mesmo com poucas variáveis.

### Conclusão

O Naive Bayes consegui manter um desempenho consistente ao longo das experiências. As alterações relacionadas a random_state e redução de atriutos, tiveram um impacto reduzido, Por outro lado, o acrescento de variáveis categóricas, provocou uma ligeira degradação do desempenho.

Assim podemos concluir que a preparação dos dados e a seleção de atributos possuem maior impacto no desempenho do Naive Bayes do que alterações de aleatoriedade ou tratamento adicional de missing values.